# 🚀 Google Colab Multi-Link Downloader (Bypass 87GB Disk Limit)

Ứng dụng tải file đa nguồn tốc độ cao trên Google Colab sử dụng **JDownloader 2 (JD2) Headless** và Direct Streaming Engine:
- **Direct-to-Drive Streaming (Bỏ qua đĩa ảo Colab)**: Dữ liệu tải được stream trực tiếp vào Google Drive (`/content/drive/MyDrive/Downloads/`) theo từng luồng tuần tự (single chunk), **không lưu trữ trung gian** trên đĩa ảo Colab.
- **Tải file không giới hạn dung lượng**: Hỗ trợ tải các file dung lượng khổng lồ **> 87GB** mà không làm đầy đĩa cục bộ Colab (dung lượng đĩa cục bộ giữ nguyên không đổi).
- **Vô hiệu hóa Pre-allocation & Sparse Files**: Ngăn chặn tình trạng tạo file rỗng chiếm hết ổ đĩa ảo hoặc lỗi hệ thống file FUSE của Google Drive.
- **Tương thích Antigravity IDE**: Hỗ trợ cả 2 chế độ: giao diện trực quan (Interactive GUI) hoặc chạy trực tiếp bằng dòng lệnh / console HTML (Direct Mode).


In [ ]:
# [1] Mount Google Drive và cài đặt môi trường Java JRE + myjdapi
import os, sys

# 1. Kiểm tra và Mount Google Drive
if os.path.exists('/content/drive/MyDrive') or os.path.exists('/content/drive/My Drive'):
    print("✅ Google Drive đã được mount sẵn tại /content/drive/MyDrive")
else:
    try:
        from google.colab import drive
        print("📂 Đang mount Google Drive...")
        drive.mount('/content/drive', timeout_ms=30000)
        print("✅ Google Drive đã sẵn sàng!")
    except Exception as e:
        print(f"⚠️ Lưu ý mount Google Drive trên Antigravity IDE: {e}")
        print("👉 Bạn cũng có thể mở Command Palette (Ctrl+Shift+P) -> gõ 'Colab: Mount Google Drive to Server'")

# 2. Cài đặt các gói công cụ và thư viện cần thiết
!apt-get update -qq && apt-get install -y -qq default-jre aria2 megatools
!pip install -q --upgrade myjdapi ipywidgets requests


In [ ]:
# [2] Khởi chạy Giao diện Tải file Đa nguồn JDownloader 2 (Direct-to-Drive)

"""
Google Colab Multi-Link Downloader with JDownloader 2 Headless Engine
Supports 1000+ file hosting providers with an interactive ipywidgets GUI.
Automatically configures Headless LinkgrabberSettings (auto-confirm & auto-start)
and works both WITHOUT a MyJDownloader account (Local API / FolderWatch) and WITH Cloud API.
"""

import os
import re
import sys
import time
import json
import shutil
import zipfile
import asyncio
import subprocess
import threading
import urllib.parse
from typing import List, Optional, Tuple, Dict, Any

# Check and import requests
try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests

# Check and import myjdapi
try:
    import myjdapi
    from myjdapi import Myjdapi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "myjdapi"])
    import myjdapi
    from myjdapi import Myjdapi

# Check and import ipywidgets & IPython
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML

# Detect Google Colab environment
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IS_COLAB:
    COLAB_ROOT = "/content"
    JD_INSTALL_DIR = os.path.join(COLAB_ROOT, "jdownloader")
    TEMP_DOWNLOAD_DIR = os.path.join(COLAB_ROOT, "temp_downloads")
    DRIVE_MOUNT_POINT = os.path.join(COLAB_ROOT, "drive")
    DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, "MyDrive")
    if not os.path.exists(DRIVE_BASE) and os.path.exists(os.path.join(DRIVE_MOUNT_POINT, "My Drive")):
        DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, "My Drive")
    DRIVE_DOWNLOADS = os.path.join(DRIVE_BASE, "Downloads")
else:
    COLAB_ROOT = os.path.abspath("./colab_env")
    JD_INSTALL_DIR = os.path.join(COLAB_ROOT, "jdownloader")
    TEMP_DOWNLOAD_DIR = os.path.join(COLAB_ROOT, "temp_downloads")
    DRIVE_BASE = os.path.join(COLAB_ROOT, "drive", "MyDrive")
    DRIVE_DOWNLOADS = os.path.join(DRIVE_BASE, "Downloads")

# Persistent configuration directory in Google Drive (survives Colab resets)
DRIVE_JD_DIR = os.path.join(DRIVE_BASE, "JDownloader")
DRIVE_JD_CFG = os.path.join(DRIVE_JD_DIR, "cfg")
DRIVE_MYJD_SETTINGS = os.path.join(DRIVE_JD_CFG, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json")
DRIVE_CRED_BACKUP = os.path.join(DRIVE_JD_DIR, "myjd_credentials.json")

os.makedirs(TEMP_DOWNLOAD_DIR, exist_ok=True)
os.makedirs(JD_INSTALL_DIR, exist_ok=True)
os.makedirs(DRIVE_DOWNLOADS, exist_ok=True)
try:
    os.makedirs(DRIVE_JD_CFG, exist_ok=True)
except Exception:
    pass


# --- UTILITY HELPERS ---
def format_size(bytes_val: int) -> str:
    """Format bytes into human-readable string."""
    if not bytes_val or bytes_val <= 0:
        return "0.0 B"
    val = float(bytes_val)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if val < 1024.0:
            return f"{val:.1f} {unit}"
        val /= 1024.0
    return f"{val:.1f} PB"


def format_eta(seconds: int) -> str:
    """Format ETA seconds into mm:ss or hh:mm:ss."""
    if not seconds or seconds <= 0:
        return "--:--"
    seconds = int(seconds)
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h:d}h {m:02d}m {s:02d}s"
    return f"{m:02d}m {s:02d}s"


def sanitize_filename(filename: str) -> str:
    """Sanitize filename to prevent invalid characters across filesystems."""
    if not filename:
        return f"download_{int(time.time())}"
    cleaned = re.sub(r'[\\/*?:"<>|]', "_", filename).strip()
    return cleaned if cleaned else f"download_{int(time.time())}"


def safe_print(msg: str, end: str = "\n"):
    """Safely print strings with emojis on consoles without charmap encode errors."""
    try:
        print(msg, end=end, flush=True)
    except (UnicodeEncodeError, UnicodeError):
        try:
            print(msg.encode('ascii', errors='backslashreplace').decode('ascii'), end=end, flush=True)
        except Exception:
            pass


def get_colab_disk_info() -> Dict[str, Any]:
    """Get current Colab local root disk usage in real time."""
    try:
        total, used, free = shutil.disk_usage('/')
        return {
            "total_gb": round(total / (1024 ** 3), 2),
            "used_gb": round(used / (1024 ** 3), 2),
            "free_gb": round(free / (1024 ** 3), 2)
        }
    except Exception:
        return {"total_gb": 0.0, "used_gb": 0.0, "free_gb": 0.0}


def save_credentials_to_drive(email: str, password: str, device_name: str = "Colab_Downloader") -> bool:
    """Save MyJDownloader credentials permanently to Google Drive and local JDownloader cfg."""
    data = {
        "email": email.strip(),
        "password": password.strip(),
        "devicename": device_name.strip(),
        "autoconnectenabledv2": True,
        "directconnectmode": "LAN_WAN_MANUAL"
    }
    backup_data = {
        "email": email.strip(),
        "password": password.strip(),
        "devicename": device_name.strip(),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    try:
        os.makedirs(DRIVE_JD_CFG, exist_ok=True)
        with open(DRIVE_MYJD_SETTINGS, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        with open(DRIVE_CRED_BACKUP, "w", encoding="utf-8") as f:
            json.dump(backup_data, f, indent=2)
    except Exception as e:
        safe_print(f"⚠️ Note saving credentials to Drive: {e}")

    try:
        local_cfg = os.path.join(JD_INSTALL_DIR, "cfg")
        os.makedirs(local_cfg, exist_ok=True)
        with open(os.path.join(local_cfg, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json"), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
    except Exception as e:
        safe_print(f"⚠️ Note saving credentials locally: {e}")

    return True


def load_credentials_from_drive() -> Dict[str, str]:
    """Load saved credentials from Google Drive (if available)."""
    for p in [DRIVE_CRED_BACKUP, DRIVE_MYJD_SETTINGS]:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if data.get("email") and data.get("password"):
                        return {
                            "email": str(data.get("email", "")).strip(),
                            "password": str(data.get("password", "")).strip(),
                            "devicename": str(data.get("devicename", "Colab_Downloader")).strip()
                        }
            except Exception:
                pass
    return {"email": "", "password": "", "devicename": "Colab_Downloader"}


# --- LOCAL JDOWNLOADER 2 ADAPTER (WORKS WITHOUT MYJDOWNLOADER ACCOUNT) ---
class LocalLinkgrabber:
    def __init__(self, adapter: "LocalJD2Device"):
        self.adapter = adapter

    def clear_list(self):
        return self.adapter.call("/linkgrabberv2/clearList")

    def add_links(self, params: List[Dict[str, Any]]):
        res = self.adapter.call("/linkgrabberv2/addLinks", params)
        if res is None and params:
            # Fallback to Click'n'Load (port 9666) + FolderWatch (.crawljob)
            p = params[0]
            links_str = p.get("links", "")
            autostart = bool(p.get("autostart", True))
            dest = p.get("destinationFolder", DRIVE_DOWNLOADS)
            pkg = p.get("packageName", "Colab_Download")
            self.adapter.add_via_local_fallbacks(links_str, dest, pkg, autostart=autostart)
        return res

    def is_collecting(self) -> bool:
        res = self.adapter.call("/linkgrabberv2/isCollecting")
        return bool(res) if res is not None else False

    def query_links(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{
            "bytesTotal": True,
            "status": True,
            "enabled": True,
            "hosts": True,
            "url": True,
            "availability": True
        }]
        res = self.adapter.call("/linkgrabberv2/queryLinks", q)
        if isinstance(res, list) and res:
            return res
        return self.adapter.read_cfg_zip_items("linkcollector")

    def query_packages(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{"bytesTotal": True, "childCount": True}]
        res = self.adapter.call("/linkgrabberv2/queryPackages", q)
        return res if isinstance(res, list) else []

    def move_to_downloadlist(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/linkgrabberv2/moveToDownloadlist", [link_ids or [], package_ids or []])

    def remove_links(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/linkgrabberv2/removeLinks", [link_ids or [], package_ids or []])


class LocalDownloads:
    def __init__(self, adapter: "LocalJD2Device"):
        self.adapter = adapter

    def query_links(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{
            "bytesTotal": True,
            "bytesLoaded": True,
            "speed": True,
            "eta": True,
            "finished": True,
            "running": True,
            "skipped": True,
            "status": True,
            "extractionStatus": True,
            "enabled": True,
            "host": True,
            "url": True
        }]
        res = self.adapter.call("/downloadsV2/queryLinks", q)
        if isinstance(res, list) and res:
            return res
        return self.adapter.read_cfg_zip_items("downloadList")

    def query_packages(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{
            "bytesTotal": True,
            "bytesLoaded": True,
            "speed": True,
            "eta": True,
            "finished": True,
            "running": True,
            "status": True,
            "childCount": True,
            "saveTo": True
        }]
        res = self.adapter.call("/downloadsV2/queryPackages", q)
        return res if isinstance(res, list) else []

    def force_download(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/downloadsV2/forceDownload", [link_ids or [], package_ids or []])

    def remove_links(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/downloadsV2/removeLinks", [link_ids or [], package_ids or []])


class LocalDownloadController:
    def __init__(self, adapter: "LocalJD2Device"):
        self.adapter = adapter

    def start_downloads(self) -> bool:
        res = self.adapter.call("/downloadcontroller/start")
        return bool(res) if res is not None else True

    def stop_downloads(self) -> bool:
        res = self.adapter.call("/downloadcontroller/stop")
        return bool(res) if res is not None else True

    def force_download(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/downloadcontroller/forceDownload", [link_ids or [], package_ids or []])

    def get_current_state(self) -> str:
        res = self.adapter.call("/downloadcontroller/getCurrentState")
        return str(res) if res else "RUNNING"

    def get_speed_in_bytes(self) -> int:
        res = self.adapter.call("/downloadcontroller/getSpeedInBytes")
        try:
            return max(0, int(res or 0))
        except Exception:
            return 0


class LocalJD2Device:
    """
    Communicates directly with local JDownloader 2 Headless instance on 127.0.0.1
    without requiring a MyJDownloader account.
    Supports:
    - Deprecated RemoteAPI (http://127.0.0.1:3128)
    - FlashGot / Click'n'Load interface (http://127.0.0.1:9666)
    - FolderWatch (.crawljob) + cfg/ zip state inspection
    """

    def __init__(self, jd_dir: str = JD_INSTALL_DIR, port: int = 3128):
        self.jd_dir = jd_dir
        self.cfg_dir = os.path.join(jd_dir, "cfg")
        self.port = port
        self.base_url = f"http://127.0.0.1:{port}"
        self.cnl_url = "http://127.0.0.1:9666"
        self.linkgrabber = LocalLinkgrabber(self)
        self.downloads = LocalDownloads(self)
        self.downloadcontroller = LocalDownloadController(self)

    def is_alive(self) -> bool:
        # 1. Check port 3128 (Deprecated RemoteAPI)
        try:
            r = requests.get(f"{self.base_url}/jd/version", timeout=1.5)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        # 2. Check port 9666 (Click'n'Load local server)
        try:
            r = requests.get(f"{self.cnl_url}/jdcheck.js", timeout=1.5)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        return False

    def call(self, endpoint: str, params: Optional[List[Any]] = None) -> Any:
        url = f"{self.base_url}{endpoint}"
        try:
            if not params:
                resp = requests.get(url, timeout=4)
            else:
                query_params = {}
                for i, p in enumerate(params):
                    query_params[f"param{i}"] = json.dumps(p)
                resp = requests.post(url, data=query_params, timeout=5)
                if resp.status_code != 200:
                    qs = "&".join(f"{k}={urllib.parse.quote(v)}" for k, v in query_params.items())
                    resp = requests.get(f"{url}?{qs}", timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                return data.get("data", data) if isinstance(data, dict) else data
        except Exception:
            pass
        return None

    def add_via_local_fallbacks(self, links_str: str, dest_folder: str, package_name: str, autostart: bool = True):
        """Add links via port 9666 Click'n'Load and FolderWatch .crawljob when port 3128 is unavailable."""
        # 1. Port 9666 Click'n'Load / FlashGot
        try:
            payload = {
                "urls": links_str,
                "dir": dest_folder,
                "package": package_name,
                "autostart": "1" if autostart else "0"
            }
            requests.post(f"{self.cnl_url}/flashgot", data=payload, timeout=3)
        except Exception:
            pass

        # 2. FolderWatch .crawljob file
        try:
            fw_dir = os.path.join(self.jd_dir, "folderwatch")
            os.makedirs(fw_dir, exist_ok=True)
            job_file = os.path.join(fw_dir, f"job_{int(time.time() * 1000)}.crawljob")
            bool_str = "TRUE" if autostart else "FALSE"
            content = (
                f"text={links_str}\n"
                f"downloadFolder={dest_folder}\n"
                f"packageName={package_name}\n"
                f"autoStart={bool_str}\n"
                f"autoConfirm={bool_str}\n"
                f"forcedStart={bool_str}\n"
                f"enabled=TRUE\n"
                f"extractAfterDownload=FALSE\n"
            )
            with open(job_file, "w", encoding="utf-8") as f:
                f.write(content)
        except Exception:
            pass

        # 3. CLI -add-links fallback
        jar_path = os.path.join(self.jd_dir, "JDownloader.jar")
        if shutil.which("java") and os.path.exists(jar_path):
            try:
                urls = [u.strip() for u in links_str.splitlines() if u.strip()]
                if urls:
                    subprocess.Popen(
                        ["java", "-Djava.awt.headless=true", "-jar", jar_path, "-add-links"] + urls,
                        cwd=self.jd_dir,
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL
                    )
            except Exception:
                pass

    def read_cfg_zip_items(self, prefix: str = "downloadList") -> List[Dict[str, Any]]:
        """Read JD2's latest downloadList*.zip or linkcollector*.zip from cfg/ directory."""
        if not os.path.exists(self.cfg_dir):
            return []
        try:
            zips = [
                os.path.join(self.cfg_dir, f)
                for f in os.listdir(self.cfg_dir)
                if f.startswith(prefix) and f.endswith(".zip")
            ]
            if not zips:
                return []
            latest_zip = max(zips, key=os.path.getmtime)
            items = []
            with zipfile.ZipFile(latest_zip, "r") as zf:
                for name in sorted(zf.namelist()):
                    if "_" in name:  # Link entry inside package
                        try:
                            raw = json.loads(zf.read(name).decode("utf-8", errors="ignore"))
                            fname = raw.get("name") or "File"
                            b_total = max(0, int(raw.get("size", 0) or 0))
                            dl_state = str(raw.get("finalLinkState") or "")
                            is_fin = "FINISHED" in dl_state.upper()
                            items.append({
                                "uuid": raw.get("uid", int(time.time())),
                                "packageUUID": raw.get("parentID", 0),
                                "name": fname,
                                "bytesTotal": b_total,
                                "bytesLoaded": b_total if is_fin else 0,
                                "finished": is_fin,
                                "running": not is_fin,
                                "status": dl_state or "Downloading",
                                "url": raw.get("url", "")
                            })
                        except Exception:
                            pass
            return items
        except Exception:
            return []


# --- JDOWNLOADER 2 HEADLESS SERVICE MANAGER ---
class JD2Service:
    """Manages installation, configuration, daemon process, and connection for JDownloader 2."""

    def __init__(self, jd_dir: str = JD_INSTALL_DIR, port: int = 3128):
        self.jd_dir = jd_dir
        self.port = port
        self.cfg_dir = os.path.join(jd_dir, "cfg")
        self.jar_path = os.path.join(jd_dir, "JDownloader.jar")
        self.jd_api = Myjdapi()
        self.jd_api.set_app_key("ColabDownloaderApp")
        self.local_device = LocalJD2Device(jd_dir=jd_dir, port=port)
        self.device: Any = None
        self.process: Optional[subprocess.Popen] = None
        self._connected = False

    def is_installed(self) -> bool:
        return os.path.exists(self.jar_path) and os.path.getsize(self.jar_path) > 1024 * 100

    def install_prerequisites(self, status_callback: Optional[callable] = None):
        """Ensure Java JRE is installed on system."""
        if shutil.which("java") is None and IS_COLAB:
            if status_callback:
                status_callback("📦 Đang cài đặt Java JRE (default-jre) cho JDownloader 2...")
            try:
                subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                subprocess.run(["apt-get", "install", "-y", "-qq", "default-jre"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception as e:
                safe_print(f"⚠️ Warning installing Java: {e}")

    def download_jdownloader_jar(self, status_callback: Optional[callable] = None):
        """Download official JDownloader.jar if missing."""
        if not self.is_installed():
            if status_callback:
                status_callback("⬇️ Đang tải JDownloader.jar...")
            url = "http://installer.jdownloader.org/JDownloader.jar"
            try:
                resp = requests.get(url, timeout=30, stream=True)
                resp.raise_for_status()
                with open(self.jar_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            except Exception as e:
                raise RuntimeError(f"Failed to download JDownloader.jar: {e}")

    def configure_settings(
        self,
        email: str = "",
        password: str = "",
        device_name: str = "Colab_Downloader",
        download_folder: Optional[str] = None,
        auto_confirm: bool = True,
        auto_start: bool = True
    ):
        """
        Write configuration JSON files BEFORE starting JDownloader 2:
        1. LinkgrabberSettings.json:
           - linkgrabberautoconfirmenabled = True
           - linkgrabberautostartenabled = True
        2. GeneralSettings.json (Direct Google Drive FUSE streaming)
        3. RemoteAPIConfig.json (Local API on 127.0.0.1:3128 & Click'n'Load 9666)
        4. FolderWatchExtension.json (Enable local .crawljob watching)
        5. MyJDownloaderSettings.json (Optional Cloud control)
        """
        os.makedirs(self.cfg_dir, exist_ok=True)
        target_download_folder = download_folder or DRIVE_DOWNLOADS
        os.makedirs(target_download_folder, exist_ok=True)

        # 1. Enable Headless Auto-Confirm & Auto-Start in LinkgrabberSettings (Theo gợi ý trong ảnh)
        lg_cfg_name = "org.jdownloader.gui.views.linkgrabber.addlinksdialog.LinkgrabberSettings.json"
        lg_data = {
            "linkgrabberautoconfirmenabled": bool(auto_confirm),
            "linkgrabberautostartenabled": bool(auto_start),
            "autoconfirmmanagerautostart": "ENABLED",
            "autostart": bool(auto_start),
            "autoconfirmdelay": 1000,
            "handledupes": True,
            "autoextractionenabled": False,
            "linkgrabberaddattop": False
        }
        for cfg_folder in [self.cfg_dir, DRIVE_JD_CFG]:
            try:
                os.makedirs(cfg_folder, exist_ok=True)
                with open(os.path.join(cfg_folder, lg_cfg_name), "w", encoding="utf-8") as f:
                    json.dump(lg_data, f, indent=2)
            except Exception:
                pass

        # 2. Enable Local RemoteAPI on port 3128 & Click'n'Load
        remote_api_cfg = os.path.join(self.cfg_dir, "org.jdownloader.api.RemoteAPIConfig.json")
        remote_api_data = {
            "deprecatedapienabled": True,
            "deprecatedapilocalhostonly": True,
            "deprecatedapiport": self.port,
            "externinterfaceenabled": True
        }
        with open(remote_api_cfg, "w", encoding="utf-8") as f:
            json.dump(remote_api_data, f, indent=2)

        # 3. Configure General Download Settings for Google Drive FUSE streaming
        general_cfg = os.path.join(self.cfg_dir, "org.jdownloader.settings.GeneralSettings.json")
        general_data = {
            "defaultdownloadfolder": target_download_folder,
            "autoextractenabled": False,
            "maxsimultanousdownloads": 1,
            "maxchunksperserver": 1,
            "maxchunksperdownload": 1,
            "createpreallocatedlargefiles": False,
            "createpreallocatedlargefilesenabled": False,
            "useallocatesparsefile": False,
            "allocatesparsefiles": False,
            "flushbufferlevel": 100,
            "maxbuffersize": 500,
            "autostartdownloadsonstartupenabled": True,
            "iffileexistsaction": "OVERWRITE_FILE"
        }
        with open(general_cfg, "w", encoding="utf-8") as f:
            json.dump(general_data, f, indent=2)

        # 4. Enable FolderWatch Extension for local .crawljob support
        fw_cfg = os.path.join(self.cfg_dir, "org.jdownloader.extensions.folderwatchV2.FolderWatchExtension.json")
        try:
            with open(fw_cfg, "w", encoding="utf-8") as f:
                json.dump({"enabled": True}, f, indent=2)
        except Exception:
            pass

        # 5. Optional MyJDownloader Cloud Settings
        if email and password:
            myjd_cfg = os.path.join(self.cfg_dir, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json")
            myjd_data = {
                "email": email.strip(),
                "password": password.strip(),
                "devicename": device_name.strip(),
                "autoconnectenabledv2": True,
                "directconnectmode": "LAN_WAN_MANUAL"
            }
            with open(myjd_cfg, "w", encoding="utf-8") as f:
                json.dump(myjd_data, f, indent=2)

    def _spawn_daemon(self):
        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        cmd = ["java", "-Djava.awt.headless=true", "-jar", self.jar_path]
        try:
            self._log_file = open(log_path, "a", encoding="utf-8")
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=self._log_file,
                stderr=subprocess.STDOUT
            )
        except Exception:
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

    def restart_service_with_cloud(self, email: str = "", password: str = "", device_name: str = "Colab_Downloader", status_callback: Optional[callable] = None) -> bool:
        """Configure credentials/settings and restart JDownloader 2 daemon."""
        self.configure_settings(email=email, password=password, device_name=device_name, auto_confirm=True, auto_start=True)

        if self.process:
            try:
                self.process.terminate()
                self.process.wait(timeout=3)
            except Exception:
                try:
                    self.process.kill()
                except Exception:
                    pass
            self.process = None

        if IS_COLAB:
            try:
                subprocess.run(["pkill", "-f", "JDownloader.jar"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception:
                pass

        self.install_prerequisites(status_callback)
        self.download_jdownloader_jar(status_callback)

        if shutil.which("java") is None:
            if not IS_COLAB:
                return True
            raise RuntimeError("Java is required to run JDownloader 2.")

        if status_callback:
            status_callback("🚀 Đang khởi động JDownloader 2 với cấu hình Auto-Confirm & Auto-Start...")

        self._spawn_daemon()
        return True

    def start_service(self, email: str = "", password: str = "", device_name: str = "Colab_Downloader", status_callback: Optional[callable] = None) -> bool:
        """Start JDownloader 2 in background and connect via Cloud API (if configured) or Local API."""
        # 1. Check if Local API is already alive
        if self.local_device.is_alive():
            self.device = self.local_device
            self._connected = True
            if email and password:
                self.connect_myjd(email, password, device_name, timeout=4)
            if status_callback:
                status_callback("✅ Đã kết nối với JDownloader 2 đang chạy.")
            return True

        # 2. Check if Cloud API is already connected
        if email and password and self.connect_myjd(email, password, device_name, timeout=4):
            if status_callback:
                status_callback("✅ Đã kết nối JDownloader 2 qua MyJDownloader Cloud.")
            return True

        if shutil.which("java") is None:
            if not IS_COLAB:
                self.device = self.local_device
                self._connected = True
                return True
            raise RuntimeError("Java is required to run JDownloader 2.")

        if status_callback:
            status_callback("🚀 Đang khởi chạy JDownloader 2 Headless Service (lần đầu có thể mất ~20-40s để cập nhật Core)...")

        self._spawn_daemon()

        # Poll connection to Local API or Cloud API
        start_time = time.time()
        timeout = 75
        while time.time() - start_time < timeout:
            time.sleep(2)
            # If JD2 exited after first-run self-update, re-launch it automatically
            if self.process and self.process.poll() is not None:
                self._spawn_daemon()

            # Check Local API (works without account)
            if self.local_device.is_alive():
                self.device = self.local_device
                self._connected = True
                if email and password:
                    self.connect_myjd(email, password, device_name, timeout=3)
                if status_callback:
                    status_callback("✅ JDownloader 2 đã kết nối thành công!")
                return True

            # Check MyJDownloader Cloud API if credentials provided
            if email and password:
                if self.connect_myjd(email, password, device_name, timeout=2):
                    if status_callback:
                        status_callback("✅ JDownloader 2 đã kết nối thành công qua MyJDownloader!")
                    return True

        # Fallback: even if HTTP ports are still warming up, use local_device (FolderWatch/CLI + File Monitor)
        self.device = self.local_device
        self._connected = True
        if status_callback:
            status_callback("✅ JDownloader 2 Headless Engine đã sẵn sàng (Local Mode).")
        return True

    def get_service_logs(self, max_lines: int = 15) -> str:
        """Get recent log output from running JDownloader 2 service."""
        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        if not os.path.exists(log_path):
            return "No log file found yet."
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as f:
                lines = f.readlines()
                return "".join(lines[-max_lines:]) if lines else "Log is empty."
        except Exception as e:
            return f"Error reading log: {e}"

    def connect_myjd(self, email: str, password: str, device_name: str = "Colab_Downloader", timeout: int = 12) -> bool:
        """Connect via MyJDownloader Cloud if credentials are supplied."""
        if not email or not password:
            return False
        try:
            self.jd_api.connect(email, password)
            start_t = time.time()
            while time.time() - start_t < timeout:
                try:
                    self.jd_api.update_devices()
                    dev = self.jd_api.get_device(device_name)
                    if dev:
                        self.device = dev
                        self._connected = True
                        return True
                except Exception:
                    pass
                time.sleep(1.5)
        except Exception:
            pass
        return False


# --- GOOGLE DRIVE FILE TRANSFER ---
def transfer_to_google_drive(
    local_path: str,
    drive_folder: str = DRIVE_DOWNLOADS,
    progress_callback: Optional[callable] = None
) -> Tuple[bool, str]:
    """Safely transfers a completed file from Colab local disk to Google Drive."""
    if not os.path.exists(local_path):
        return False, f"Source file does not exist: {local_path}"

    os.makedirs(drive_folder, exist_ok=True)
    filename = os.path.basename(local_path)
    dest_path = os.path.join(drive_folder, filename)

    if os.path.exists(dest_path):
        base, ext = os.path.splitext(filename)
        dest_path = os.path.join(drive_folder, f"{base}_{int(time.time())}{ext}")

    file_size = os.path.getsize(local_path)
    copied_bytes = 0
    buffer_size = 16 * 1024 * 1024  # 16 MB

    if progress_callback:
        progress_callback(0, f"Transferring to Google Drive: {os.path.basename(dest_path)}")

    try:
        with open(local_path, "rb") as fsrc, open(dest_path, "wb") as fdst:
            while True:
                chunk = fsrc.read(buffer_size)
                if not chunk:
                    break
                fdst.write(chunk)
                copied_bytes += len(chunk)
                if progress_callback and file_size > 0:
                    pct = (copied_bytes / file_size) * 100.0
                    progress_callback(pct, f"Moving to Drive: {pct:.1f}% ({format_size(copied_bytes)} / {format_size(file_size)})")

        if os.path.exists(dest_path) and os.path.getsize(dest_path) == file_size:
            os.remove(local_path)
            return True, dest_path
        else:
            return False, "File size mismatch after moving to Google Drive"
    except Exception as e:
        return False, f"Transfer error: {str(e)}"


# --- IPYWIDGETS GUI APPLICATION ---
class ColabDownloaderApp:
    """Builds and manages the interactive UI with JDownloader 2 backend."""

    def __init__(self, jd_service: Optional[JD2Service] = None):
        self.jd = jd_service or JD2Service()
        self.pending_items: List[Dict[str, Any]] = []
        self.sort_ascending = True
        self.is_downloading = False
        self._cancel_requested = False
        self._initial_drive_snapshot: Dict[str, int] = {}
        self._last_speed_check_t: float = 0.0
        self._last_speed_bytes: int = 0
        self._computed_speed_bps: int = 0

        # Custom CSS for Dark Jupyter / Colab theme
        self.css_styles = widgets.HTML("""
        <style>
            .colab-dl-container {
                background-color: #1e1e1e;
                color: #e0e0e0;
                padding: 16px;
                border-radius: 8px;
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            }
            .widget-textarea textarea {
                background-color: #2d2d2d !important;
                color: #f1f1f1 !important;
                border: 1px solid #444 !important;
                border-radius: 4px !important;
                font-family: monospace !important;
                font-size: 13px !important;
            }
            .widget-textarea textarea::placeholder {
                color: #888 !important;
            }
            .widget-select-multiple select {
                background-color: #252526 !important;
                color: #d4d4d4 !important;
                border: 1px solid #3c3c3c !important;
                font-family: monospace !important;
                font-size: 13px !important;
            }
            .widget-select-multiple select option:checked {
                background-color: #04395e !important;
                color: #ffffff !important;
            }
            .queue-header {
                font-size: 14px;
                font-weight: bold;
                color: #f0f0f0;
                margin-bottom: 6px;
            }
        </style>
        """)

        # 1. Links Textarea
        self.text_area = widgets.Textarea(
            description='Links:',
            placeholder='Paste Links Here (e.g., Vikingfile, Rapidgator, Mega, 1fichier, Yandex)...',
            layout=widgets.Layout(width='98%', height='140px')
        )

        # 2. Main Action Buttons (Removed Sample Link, Added Cancel Download)
        self.btn_resolve = widgets.Button(
            description="Resolve Links",
            button_style='success',
            icon='search',
            tooltip='Kiểm tra link với JDownloader 2 và hiển thị danh sách chờ (Queue Preview)',
            layout=widgets.Layout(width='145px', height='36px')
        )
        self.btn_quick = widgets.Button(
            description="Quick Download",
            button_style='primary',
            icon='bolt',
            tooltip='Tự động thêm vào LinkGrabber và tải thẳng vào Google Drive ngay lập tức',
            layout=widgets.Layout(width='155px', height='36px')
        )
        self.btn_cancel_dl = widgets.Button(
            description="Cancel Download",
            button_style='danger',
            icon='stop',
            tooltip='Dừng quá trình tải/kiểm tra hiện tại và khôi phục nút Resolve Links',
            layout=widgets.Layout(width='155px', height='36px')
        )

        # 3. Queue Preview Container & Controls
        self.queue_header = widgets.HTML(
            "<div class='queue-header'>📄 <b>Queue Preview</b> <span style='font-weight:normal; color:#aaa;'>(Select items to manage)</span></div>"
        )
        self.queue_list = widgets.SelectMultiple(
            description='Queue:',
            options=[],
            layout=widgets.Layout(width='98%', height='160px')
        )

        # Toolbar Buttons (Under Queue Preview)
        self.btn_up = widgets.Button(description="▲ Up", layout=widgets.Layout(width='80px'))
        self.btn_down = widgets.Button(description="▼ Down", layout=widgets.Layout(width='80px'))
        self.btn_sort = widgets.Button(description="Sort A-Z", layout=widgets.Layout(width='90px'))
        self.btn_select_all = widgets.Button(description="Select All", layout=widgets.Layout(width='85px'))
        self.btn_none = widgets.Button(description="None", layout=widgets.Layout(width='65px'))
        self.btn_start = widgets.Button(
            description="▶ Start Download",
            button_style='success',
            layout=widgets.Layout(width='145px')
        )
        self.btn_cancel = widgets.Button(
            description="Cancel",
            button_style='warning',
            layout=widgets.Layout(width='75px')
        )
        self.btn_remove = widgets.Button(
            description="Remove",
            button_style='danger',
            layout=widgets.Layout(width='80px')
        )

        self.queue_toolbar = widgets.HBox([
            self.btn_up,
            self.btn_down,
            self.btn_sort,
            self.btn_select_all,
            self.btn_none,
            self.btn_start,
            self.btn_cancel,
            self.btn_remove
        ], layout=widgets.Layout(margin='8px 0 0 0'))

        self.queue_container = widgets.VBox([
            self.queue_header,
            self.queue_list,
            self.queue_toolbar
        ], layout=widgets.Layout(display='none', margin='12px 0'))

        # 4. Progress and Status Elements
        self.progress_bar = widgets.FloatProgress(
            value=0.0,
            min=0.0,
            max=100.0,
            description='Idle',
            bar_style='info',
            style={'description_width': '160px'},
            layout=widgets.Layout(width='98%', margin='8px 0')
        )
        self.status_label = widgets.HTML(value="")

        # Real-time Colab Disk Usage Widget
        self.disk_info_widget = widgets.HTML(value=self._get_disk_html())

        # Auto-load saved credentials from Google Drive
        saved_creds = load_credentials_from_drive()
        saved_email = saved_creds.get("email", "")
        saved_pwd = saved_creds.get("password", "")

        # Optional MyJDownloader Credentials Accordion (collapsed by default)
        self.myjd_email = widgets.Text(
            description='Email:',
            value=saved_email,
            placeholder='my.jdownloader.org email (Tùy chọn)',
            layout=widgets.Layout(width='260px')
        )
        self.myjd_pass = widgets.Password(
            description='Password:',
            value=saved_pwd,
            placeholder='Password',
            layout=widgets.Layout(width='220px')
        )
        self.btn_login = widgets.Button(
            description='Login & Lưu Drive' if not saved_email else 'Login lại (Drive)',
            button_style='success' if saved_email else 'primary',
            icon='check' if saved_email else 'sign-in',
            tooltip='Đăng nhập và tự động lưu thông tin vào Google Drive (không bắt buộc)',
            layout=widgets.Layout(width='160px')
        )
        self.btn_login.on_click(self._on_login_myjd)

        self.btn_view_log = widgets.Button(
            description='Xem Log JD2',
            button_style='info',
            icon='file-text',
            tooltip='Xem log tiến trình JDownloader 2 thời gian thực',
            layout=widgets.Layout(width='120px')
        )
        self.btn_view_log.on_click(self._on_view_log)

        cred_info_html = """
        <div style='font-size:12px; color:#ccc; line-height:1.5; margin-bottom:8px;'>
            ✅ <b>Đã bật sẵn Headless Auto-Confirm & Auto-Start:</b> Tự động chuyển link từ LinkGrabber sang Download và tải ngay cả khi <b>không cần đăng nhập tài khoản MyJDownloader</b>.<br>
            💾 <b>Tài khoản MyJDownloader (Tùy chọn):</b> Nếu đăng nhập, thông tin sẽ được lưu tại <code>MyDrive/JDownloader/cfg/</code> để quản lý từ xa & giải Captcha qua <a href='https://my.jdownloader.org' target='_blank' style='color:#00e676; font-weight:bold;'>my.jdownloader.org</a>.
        </div>
        """
        cred_box = widgets.VBox([
            widgets.HTML(cred_info_html),
            widgets.HBox([self.myjd_email, self.myjd_pass, self.btn_login, self.btn_view_log])
        ])
        self.cred_accordion = widgets.Accordion(children=[cred_box])
        self.cred_accordion.set_title(0, '⚙️ Cấu hình MyJDownloader (Tùy chọn)')
        self.cred_accordion.selected_index = None

        self._bind_events()

    def _get_disk_html(self) -> str:
        """Render real-time Colab local disk space indicator."""
        info = get_colab_disk_info()
        free_gb = info.get("free_gb", 0.0)
        total_gb = info.get("total_gb", 0.0)
        used_gb = info.get("used_gb", 0.0)
        pct_free = (free_gb / total_gb * 100.0) if total_gb > 0 else 0.0
        return (
            f"<div style='margin: 4px 0 8px 0; padding: 7px 12px; background-color: #172317; border: 1px solid #2e7d32; border-radius: 4px; font-size: 13px; display: flex; align-items: center; justify-content: space-between;'>"
            f"<span>💾 <b>Dung lượng Colab khả dụng:</b> <span style='color: #00e676; font-weight: bold; font-size: 14px;'>{free_gb} GB</span> / {total_gb} GB (Đã dùng: {used_gb} GB — Trống: {pct_free:.1f}%)</span>"
            f"<span style='color: #81c784; font-size: 12px;'>⚡ Stream trực tiếp vào Google Drive (Đĩa Colab không bị giảm)</span>"
            f"</div>"
        )

    def _bind_events(self):
        """Bind UI events."""
        self.btn_resolve.on_click(self._on_resolve_links)
        self.btn_quick.on_click(self._on_quick_download)
        self.btn_cancel_dl.on_click(self._on_cancel_download)
        self.btn_up.on_click(self._on_queue_up)
        self.btn_down.on_click(self._on_queue_down)
        self.btn_sort.on_click(self._on_queue_sort)
        self.btn_select_all.on_click(self._on_queue_select_all)
        self.btn_none.on_click(self._on_queue_select_none)
        self.btn_start.on_click(self._on_queue_start)
        self.btn_cancel.on_click(self._on_queue_cancel)
        self.btn_remove.on_click(self._on_queue_remove)

    def _snapshot_drive_files(self) -> Dict[str, int]:
        """Record existing files and their sizes in DRIVE_DOWNLOADS before starting a new download."""
        snap = {}
        if os.path.exists(DRIVE_DOWNLOADS):
            try:
                for entry in os.listdir(DRIVE_DOWNLOADS):
                    p = os.path.join(DRIVE_DOWNLOADS, entry)
                    if os.path.isfile(p):
                        snap[entry] = os.path.getsize(p)
            except Exception:
                pass
        return snap

    def _get_drive_active_progress(self, expected_names: Optional[List[str]] = None) -> Tuple[int, str, List[str]]:
        """
        Inspect DRIVE_DOWNLOADS in real time for growing .part files or newly completed files.
        Returns (loaded_bytes_on_disk, active_filename, newly_completed_filenames).
        """
        loaded_on_disk = 0
        active_name = ""
        newly_done = []

        if not os.path.exists(DRIVE_DOWNLOADS):
            return 0, "", []

        try:
            for entry in os.listdir(DRIVE_DOWNLOADS):
                p = os.path.join(DRIVE_DOWNLOADS, entry)
                if not os.path.isfile(p):
                    continue
                sz = os.path.getsize(p)
                old_sz = self._initial_drive_snapshot.get(entry, -1)

                if entry.endswith(".part"):
                    loaded_on_disk += sz
                    active_name = entry[:-5]
                else:
                    # Check if this file is newly created or grew compared to snapshot before download
                    is_expected = any(
                        exp and (exp == entry or entry.startswith(os.path.splitext(exp)[0]))
                        for exp in (expected_names or [])
                    )
                    if old_sz == -1 or sz > old_sz or is_expected:
                        newly_done.append(entry)
                        loaded_on_disk += sz
                        if not active_name:
                            active_name = entry
        except Exception:
            pass

        return loaded_on_disk, active_name, newly_done

    def _on_cancel_download(self, b=None):
        """Interrupt active download or link resolving, stop JD2 tasks, and restore buttons immediately."""
        self._cancel_requested = True
        self.is_downloading = False

        # Restore buttons immediately so user is never stuck
        self.btn_resolve.disabled = False
        self.btn_quick.disabled = False
        self.btn_start.disabled = False
        self.queue_container.layout.display = 'none'
        self.pending_items = []

        # Stop JD2 downloads and clean up lists in background thread
        def _stop_jd_bg():
            try:
                if self.jd and self.jd.device:
                    try:
                        self.jd.device.downloadcontroller.stop_downloads()
                    except Exception:
                        pass
                    try:
                        self.jd.device.linkgrabber.clear_list()
                    except Exception:
                        pass
                    try:
                        links = self.jd.device.downloads.query_links() or []
                        unfinished_ids = [l.get("uuid") for l in links if not l.get("finished") and l.get("uuid")]
                        if unfinished_ids:
                            self.jd.device.downloads.remove_links(unfinished_ids, [])
                    except Exception:
                        pass
            except Exception:
                pass

        threading.Thread(target=_stop_jd_bg, daemon=True).start()

        self.progress_bar.value = 0.0
        self.progress_bar.bar_style = 'warning'
        self.progress_bar.description = "Cancelled"
        self.disk_info_widget.value = self._get_disk_html()
        self.status_label.value = (
            "<div style='margin-top:6px; padding:8px 12px; background-color:#3e2723; border:1px solid #ff7043; border-radius:4px; color:#ffccbc;'>"
            "🛑 <b>Đã hủy tác vụ tải hiện tại và khôi phục nút Resolve Links / Quick Download!</b> Bạn có thể thêm link mới ngay bây giờ."
            "</div>"
        )

    def _on_login_myjd(self, b=None):
        """Handle MyJDownloader login, save credentials permanently to Google Drive, and start cloud daemon."""
        email = self.myjd_email.value.strip()
        pwd = self.myjd_pass.value.strip()

        if not email or not pwd:
            self.status_label.value = "<span style='color:#ff9800;'>⚠️ Vui lòng nhập đầy đủ Email và Mật khẩu tài khoản my.jdownloader.org.</span>"
            return

        self.btn_login.disabled = True
        self.btn_login.description = "Đang lưu & kết nối..."
        self.status_label.value = "<span style='color:#29b6f6;'>💾 Đang lưu cấu hình vào Google Drive & Khởi động JDownloader 2...</span>"

        save_credentials_to_drive(email=email, password=pwd, device_name="Colab_Downloader")

        def update_msg(msg: str):
            self.status_label.value = f"<span style='color:#29b6f6;'>{msg}</span>"

        try:
            self.jd.restart_service_with_cloud(email=email, password=pwd, device_name="Colab_Downloader", status_callback=update_msg)
        except Exception as e:
            safe_print(f"Note restarting JD2: {e}")

        self.btn_login.disabled = False
        self.btn_login.description = "Đã lưu (Drive) ✅"
        self.btn_login.button_style = "success"
        self.btn_login.icon = "check"

        self.status_label.value = (
            "<div style='margin-top:6px; padding:10px 14px; background-color:#1e3b2b; border:1px solid #2e7d32; border-radius:4px; font-size:13px;'>"
            "💾 <b>Đã lưu thông tin đăng nhập & cấu hình Auto-Start vào Google Drive!</b><br>"
            "🚀 <b>JDownloader 2 Daemon</b> đang kết nối tới <code>my.jdownloader.org</code>..."
            "</div>"
        )

        def _verify_device_bg():
            time.sleep(3)
            if self.jd.connect_myjd(email, pwd, "Colab_Downloader", timeout=20):
                self.status_label.value = (
                    "<div style='margin-top:6px; padding:10px 14px; background-color:#1e3b2b; border:1px solid #2e7d32; border-radius:4px; font-size:13px;'>"
                    "✅ <b>Thiết bị 'Colab_Downloader' đã ONLINE trên my.jdownloader.org!</b>"
                    "</div>"
                )

        threading.Thread(target=_verify_device_bg, daemon=True).start()

    def _on_view_log(self, b=None):
        """Display recent JDownloader 2 log lines."""
        logs = self.jd.get_service_logs(max_lines=20)
        formatted_logs = logs.replace("<", "&lt;").replace(">", "&gt;")
        self.status_label.value = (
            f"<div style='margin-top:6px; padding:8px; background-color:#111; border:1px solid #444; border-radius:4px; font-family:monospace; font-size:12px; max-height:200px; overflow-y:auto; color:#a5d6a7;'>"
            f"<b>📜 JDownloader 2 Service Log (Recent):</b><br><pre style='margin:0; white-space:pre-wrap;'>{formatted_logs}</pre>"
            f"</div>"
        )

    def _ensure_service_ready(self) -> bool:
        """Make sure JDownloader 2 is initialized, configured with Headless Auto-Confirm/Auto-Start, and connected."""
        email = self.myjd_email.value.strip()
        pwd = self.myjd_pass.value.strip()

        # Always write Headless settings (LinkgrabberSettings auto-confirm & auto-start)
        self.jd.configure_settings(
            email=email,
            password=pwd,
            download_folder=DRIVE_DOWNLOADS,
            auto_confirm=True,
            auto_start=True
        )

        if self.jd._connected and self.jd.device:
            return True

        def update_msg(msg: str):
            self.status_label.value = f"<span style='color:#29b6f6;'>{msg}</span>"

        if IS_COLAB:
            if not os.path.exists(os.path.join(COLAB_ROOT, "drive", "MyDrive")) and \
               not os.path.exists(os.path.join(COLAB_ROOT, "drive", "My Drive")):
                update_msg("📂 Đang mount Google Drive (/content/drive)...")
                try:
                    from google.colab import drive
                    drive.mount('/content/drive')
                except Exception as e:
                    safe_print(f"Drive mount error: {e}")

        os.makedirs(DRIVE_DOWNLOADS, exist_ok=True)
        self.jd.install_prerequisites(update_msg)
        self.jd.download_jdownloader_jar(update_msg)
        return self.jd.start_service(email=email, password=pwd, status_callback=update_msg)

    def _extract_input_urls(self) -> List[str]:
        raw = self.text_area.value.strip()
        if not raw:
            return []
        return [line.strip() for line in raw.splitlines() if line.strip()]

    def update_queue_display(self):
        """Refresh the Queue listbox."""
        options = []
        for i, item in enumerate(self.pending_items, 1):
            name = item.get("name") or item.get("filename") or "File"
            b_total = max(0, int(item.get("bytesTotal", 0) or 0))
            size_str = format_size(b_total) if b_total > 0 else "Unknown size"
            display_text = f"{i}. 📦 {name} ({size_str}) → MyDrive/Downloads/{name}"
            options.append(display_text)

        self.queue_list.options = options
        self.queue_list.value = tuple(options)

    def _clear_old_finished_downloads(self):
        """Remove already-finished links from JD2 download list before starting a new session."""
        try:
            old_links = self.jd.device.downloads.query_links() or []
            fin_ids = [l.get("uuid") for l in old_links if l.get("finished") and l.get("uuid")]
            if fin_ids:
                self.jd.device.downloads.remove_links(fin_ids, [])
        except Exception:
            pass

    def _on_resolve_links(self, b=None):
        """Resolve links with JDownloader 2 LinkGrabber and populate Queue Preview."""
        urls = self._extract_input_urls()
        if not urls:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ Vui lòng dán ít nhất 1 đường link vào ô Links.</span>"
            return

        self._cancel_requested = False
        self.btn_resolve.disabled = True
        self.btn_quick.disabled = True
        self.progress_bar.value = 0.0
        self.progress_bar.bar_style = 'info'
        self.progress_bar.description = "Resolving..."
        self.status_label.value = "<span style='color:#29b6f6;'>🔍 Đang khởi tạo JDownloader 2 Engine...</span>"

        def _resolve_worker():
            try:
                if not self._ensure_service_ready():
                    self.status_label.value = "<span style='color:#e53935;'>❌ Không thể kết nối tới dịch vụ JDownloader 2.</span>"
                    return

                if self._cancel_requested:
                    return

                self.status_label.value = f"<span style='color:#29b6f6;'>🔍 JDownloader 2 LinkGrabber: Đang phân tích {len(urls)} link...</span>"

                # Pause controller during Resolve Preview so user can inspect before clicking Start Download
                try:
                    self.jd.device.downloadcontroller.stop_downloads()
                except Exception:
                    pass

                try:
                    self.jd.device.linkgrabber.clear_list()
                except Exception:
                    pass
                self._clear_old_finished_downloads()

                self.jd.device.linkgrabber.add_links([{
                    "autostart": False,
                    "links": "\n".join(urls),
                    "packageName": "Colab_Queue",
                    "destinationFolder": DRIVE_DOWNLOADS,
                    "overwritePackagizerRules": True
                }])

                start_wait = time.time()
                time.sleep(1.5)
                crawled = []
                while time.time() - start_wait < 40 and not self._cancel_requested:
                    try:
                        lg_links = self.jd.device.linkgrabber.query_links([{
                            "bytesTotal": True,
                            "status": True,
                            "enabled": True,
                            "hosts": True,
                            "url": True,
                            "availability": True
                        }]) or []
                        dl_links = self.jd.device.downloads.query_links() or []
                        crawled = lg_links if lg_links else dl_links
                        is_col = self.jd.device.linkgrabber.is_collecting()
                        if crawled and not is_col:
                            break
                    except Exception:
                        pass
                    time.sleep(1.5)

                if self._cancel_requested:
                    return

                # Fallback if hoster requires direct download crawling
                if not crawled:
                    for idx, u in enumerate(urls, 1):
                        guessed_name = u.rstrip("/").split("/")[-1] or f"Download_{idx}"
                        crawled.append({
                            "uuid": None,
                            "packageUUID": None,
                            "name": guessed_name,
                            "bytesTotal": 0,
                            "url": u,
                            "_raw_url": u
                        })
                else:
                    for idx, item in enumerate(crawled):
                        if not item.get("url") and idx < len(urls):
                            item["url"] = urls[idx]
                        item["_raw_url"] = item.get("url") or urls[min(idx, len(urls) - 1)]

                self.pending_items = crawled
                self.update_queue_display()
                self.queue_container.layout.display = 'block'
                self.progress_bar.description = "Ready"
                self.status_label.value = (
                    f"<span style='color:#66bb6a;'>✅ JDownloader 2 đã phân tích {len(self.pending_items)} mục. "
                    f"Bấm <b>▶ Start Download</b> để tải ngay vào Google Drive.</span>"
                )
            except Exception as e:
                if not self._cancel_requested:
                    self.status_label.value = f"<span style='color:#e53935;'>❌ Lỗi phân tích link: {str(e)[:120]}</span>"
                    self.progress_bar.description = "Error"
            finally:
                self.btn_resolve.disabled = False
                self.btn_quick.disabled = False

        threading.Thread(target=_resolve_worker, daemon=True).start()

    def _on_quick_download(self, b=None):
        """Bypass Queue Preview, add links with Auto-Confirm & Auto-Start, and start downloading immediately."""
        urls = self._extract_input_urls()
        if not urls:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ Vui lòng dán ít nhất 1 đường link vào ô Links.</span>"
            return

        self._cancel_requested = False
        self.queue_container.layout.display = 'none'
        self.btn_resolve.disabled = True
        self.btn_quick.disabled = True
        self.progress_bar.value = 0.0
        self.progress_bar.bar_style = 'info'
        self.progress_bar.description = "Starting..."
        self.status_label.value = "<span style='color:#29b6f6;'>⚡ Đang thêm vào LinkGrabber (Auto-Confirm & Auto-Start) và tải trực tiếp vào Google Drive...</span>"

        self._initial_drive_snapshot = self._snapshot_drive_files()

        # Instant direct streaming for Yandex Disk links
        if all("yandex" in u.lower() for u in urls):
            threading.Thread(target=self._stream_direct_worker, args=(urls,), daemon=True).start()
            return

        def _quick_start_worker():
            try:
                if not self._ensure_service_ready():
                    self.btn_resolve.disabled = False
                    self.btn_quick.disabled = False
                    self.status_label.value = "<span style='color:#e53935;'>❌ Không thể kết nối tới dịch vụ JDownloader 2.</span>"
                    return

                self._clear_old_finished_downloads()
                try:
                    self.jd.device.linkgrabber.clear_list()
                except Exception:
                    pass

                self.jd.device.linkgrabber.add_links([{
                    "autostart": True,
                    "autoConfirm": True,
                    "forcedStart": True,
                    "links": "\n".join(urls),
                    "packageName": "Colab_Quick",
                    "destinationFolder": DRIVE_DOWNLOADS,
                    "overwritePackagizerRules": True
                }])

                self._kick_jd2_downloads()
                self._start_download_monitor(raw_urls=urls)
            except Exception as e:
                self.status_label.value = f"<span style='color:#e53935;'>❌ Lỗi Quick Download: {str(e)[:120]}</span>"
                self.btn_resolve.disabled = False
                self.btn_quick.disabled = False

        threading.Thread(target=_quick_start_worker, daemon=True).start()

    def _on_queue_start(self, b=None):
        """Move selected items from Linkgrabber to download list and start downloading immediately."""
        selected_strings = list(self.queue_list.value)
        if not selected_strings:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ Vui lòng chọn ít nhất 1 file trong danh sách Queue.</span>"
            return

        selected_indices = [int(s.split('.')[0].strip()) - 1 for s in selected_strings]
        selected_tasks = [self.pending_items[i] for i in selected_indices if 0 <= i < len(self.pending_items)]

        link_ids = [item.get("uuid") for item in selected_tasks if item.get("uuid")]
        package_ids = list(set([item.get("packageUUID") for item in selected_tasks if item.get("packageUUID")]))
        expected_names = [item.get("name") for item in selected_tasks if item.get("name")]
        raw_urls = [item.get("_raw_url") or item.get("url") for item in selected_tasks if item.get("_raw_url") or item.get("url")]
        if not raw_urls:
            raw_urls = self._extract_input_urls()

        self._cancel_requested = False
        self.queue_container.layout.display = 'none'
        self.btn_resolve.disabled = True
        self.btn_quick.disabled = True
        self.progress_bar.value = 0.0
        self.progress_bar.bar_style = 'info'
        self.progress_bar.description = "Starting..."
        self.status_label.value = "<span style='color:#29b6f6;'>▶️ Đang chuyển tác vụ sang Danh sách tải JDownloader 2 & Khởi động luồng tải...</span>"

        self._initial_drive_snapshot = self._snapshot_drive_files()

        if raw_urls and all("yandex" in u.lower() for u in raw_urls):
            threading.Thread(target=self._stream_direct_worker, args=(raw_urls,), daemon=True).start()
            return

        def _start_worker():
            try:
                if not self._ensure_service_ready():
                    self.btn_resolve.disabled = False
                    self.btn_quick.disabled = False
                    self.status_label.value = "<span style='color:#e53935;'>❌ Không thể kết nối tới dịch vụ JDownloader 2.</span>"
                    return

                # 1. Move any existing LinkGrabber items to Download List
                if link_ids or package_ids:
                    try:
                        self.jd.device.linkgrabber.move_to_downloadlist(link_ids, package_ids)
                    except Exception:
                        try:
                            self.jd.device.linkgrabber.move_to_downloadlist(link_ids, [])
                        except Exception:
                            pass

                # 2. Check if items are in Download List; if not, add with autostart=True so LinkgrabberSettings auto-confirms & starts
                time.sleep(1.0)
                dl_links = self.jd.device.downloads.query_links() or []
                lg_links = self.jd.device.linkgrabber.query_links() or []

                if lg_links:
                    all_lg_ids = [l.get("uuid") for l in lg_links if l.get("uuid")]
                    all_pkg_ids = list(set([l.get("packageUUID") for l in lg_links if l.get("packageUUID")]))
                    try:
                        self.jd.device.linkgrabber.move_to_downloadlist(all_lg_ids, all_pkg_ids)
                    except Exception:
                        pass

                if not dl_links and not lg_links and raw_urls:
                    self.jd.device.linkgrabber.add_links([{
                        "autostart": True,
                        "autoConfirm": True,
                        "forcedStart": True,
                        "links": "\n".join(raw_urls),
                        "packageName": "Colab_Queue",
                        "destinationFolder": DRIVE_DOWNLOADS,
                        "overwritePackagizerRules": True
                    }])

                # 3. Force start DownloadController immediately
                self._kick_jd2_downloads(link_ids, package_ids)

                # 4. Start background progress monitor
                self._start_download_monitor(link_ids=link_ids, package_ids=package_ids, expected_names=expected_names, raw_urls=raw_urls)

            except Exception as e:
                self.status_label.value = f"<span style='color:#e53935;'>❌ Lỗi Start Download: {str(e)[:120]}</span>"
                self.btn_resolve.disabled = False
                self.btn_quick.disabled = False

        threading.Thread(target=_start_worker, daemon=True).start()

    def _kick_jd2_downloads(self, link_ids=None, package_ids=None):
        """Move any collected links from LinkGrabber to DownloadList and force DownloadController to start."""
        try:
            # Auto-move any items still sitting in LinkGrabber
            lg_links = self.jd.device.linkgrabber.query_links() or []
            if lg_links:
                lg_ids = [l.get("uuid") for l in lg_links if l.get("uuid")]
                lg_pkgs = list(set([l.get("packageUUID") for l in lg_links if l.get("packageUUID")]))
                if lg_ids or lg_pkgs:
                    self.jd.device.linkgrabber.move_to_downloadlist(lg_ids, lg_pkgs)
        except Exception:
            pass

        try:
            self.jd.device.downloadcontroller.start_downloads()
        except Exception:
            pass

        # Query current download links to force-start all unfinished links
        target_link_ids = list(link_ids or [])
        target_pkg_ids = list(package_ids or [])
        try:
            dl_links = self.jd.device.downloads.query_links() or []
            for l in dl_links:
                if not l.get("finished"):
                    if l.get("uuid") and l.get("uuid") not in target_link_ids:
                        target_link_ids.append(l.get("uuid"))
                    if l.get("packageUUID") and l.get("packageUUID") not in target_pkg_ids:
                        target_pkg_ids.append(l.get("packageUUID"))
        except Exception:
            pass

        if target_link_ids or target_pkg_ids:
            try:
                self.jd.device.downloadcontroller.force_download(target_link_ids, target_pkg_ids)
            except Exception:
                pass
            try:
                self.jd.device.downloads.force_download(target_link_ids, target_pkg_ids)
            except Exception:
                pass

    def _start_download_monitor(self, link_ids=None, package_ids=None, expected_names=None, raw_urls=None):
        """Start real-time download progress monitoring."""
        self.is_downloading = True
        self._last_speed_check_t = time.time()
        self._last_speed_bytes = 0
        self._computed_speed_bps = 0

        threading.Thread(
            target=self._monitor_and_transfer,
            args=(link_ids, package_ids, expected_names, raw_urls),
            daemon=True
        ).start()

    def _scan_and_transfer_completed(self, completed_files: set):
        """Scan local temporary directories for any downloaded files and transfer them to Google Drive."""
        temp_dirs = [
            os.path.join(COLAB_ROOT, "downloads"),
            os.path.join(self.jd.jd_dir, "downloads") if hasattr(self.jd, "jd_dir") else None,
            "./downloads"
        ]
        for t_dir in temp_dirs:
            if not t_dir or not os.path.exists(t_dir):
                continue
            for entry in os.listdir(t_dir):
                f_path = os.path.join(t_dir, entry)
                if os.path.isfile(f_path) and not entry.endswith(".part"):
                    success, _ = transfer_to_google_drive(f_path, DRIVE_DOWNLOADS)
                    if success:
                        completed_files.add(entry)

    def _monitor_tick(
        self,
        link_ids: Optional[List[Any]],
        package_ids: Optional[List[Any]],
        expected_names: Optional[List[str]],
        completed_files: set,
        wait_cycles: int
    ) -> Tuple[bool, int]:
        """
        Single monitoring iteration:
        - Moves any remaining LinkGrabber items into DownloadList automatically.
        - Queries JD2 DownloadList links & packages with explicit byte/speed fields.
        - Combines API byte counters with real-time .part file size in Google Drive.
        - Updates FloatProgress bar and status HTML accurately.
        """
        if self._cancel_requested or not self.is_downloading:
            return True, wait_cycles

        try:
            # 1. Check if links are still in LinkGrabber and move them automatically
            try:
                lg_links = self.jd.device.linkgrabber.query_links() or []
                if lg_links and not self.jd.device.linkgrabber.is_collecting():
                    lg_ids = [l.get("uuid") for l in lg_links if l.get("uuid")]
                    lg_pkgs = list(set([l.get("packageUUID") for l in lg_links if l.get("packageUUID")]))
                    self.jd.device.linkgrabber.move_to_downloadlist(lg_ids, lg_pkgs)
                    self._kick_jd2_downloads(lg_ids, lg_pkgs)
            except Exception:
                pass

            # 2. Query DownloadList Links and Packages with full status fields
            links = self.jd.device.downloads.query_links([{
                "bytesTotal": True,
                "bytesLoaded": True,
                "speed": True,
                "eta": True,
                "finished": True,
                "running": True,
                "skipped": True,
                "status": True,
                "extractionStatus": True,
                "enabled": True,
                "host": True,
                "url": True
            }]) or []

            packages = self.jd.device.downloads.query_packages([{
                "bytesTotal": True,
                "bytesLoaded": True,
                "speed": True,
                "eta": True,
                "finished": True,
                "running": True,
                "status": True,
                "childCount": True,
                "saveTo": True
            }]) or []

            # 3. Check real-time file/part progress directly on Google Drive FUSE
            known_names = list(expected_names or []) + [l.get("name") for l in links if l.get("name")]
            disk_loaded_bytes, disk_active_name, disk_newly_done = self._get_drive_active_progress(known_names)
            for df in disk_newly_done:
                completed_files.add(df)

            self._scan_and_transfer_completed(completed_files)
            cur_disk = get_colab_disk_info()
            self.disk_info_widget.value = self._get_disk_html()

            # 4. Calculate safe byte totals (avoiding -1 values from JD2 API)
            api_total_bytes = sum(max(0, int(l.get("bytesTotal") or 0)) for l in links)
            pkg_total_bytes = sum(max(0, int(p.get("bytesTotal") or 0)) for p in packages)
            total_bytes = max(api_total_bytes, pkg_total_bytes)
            if total_bytes == 0 and self.pending_items:
                total_bytes = sum(max(0, int(it.get("bytesTotal") or 0)) for it in self.pending_items)

            api_loaded_bytes = sum(max(0, int(l.get("bytesLoaded") or 0)) for l in links)
            pkg_loaded_bytes = sum(max(0, int(p.get("bytesLoaded") or 0)) for p in packages)
            loaded_bytes = max(api_loaded_bytes, pkg_loaded_bytes, disk_loaded_bytes)
            if total_bytes > 0 and loaded_bytes > total_bytes:
                loaded_bytes = total_bytes

            # 5. Compute download speed (API speed or delta bytes on disk)
            now_t = time.time()
            dt = now_t - self._last_speed_check_t
            if dt >= 1.0:
                delta_b = max(0, loaded_bytes - self._last_speed_bytes)
                self._computed_speed_bps = int(delta_b / dt)
                self._last_speed_check_t = now_t
                self._last_speed_bytes = loaded_bytes

            api_speed = 0
            try:
                api_speed = max(0, int(self.jd.device.downloadcontroller.get_speed_in_bytes() or 0))
            except Exception:
                pass
            if api_speed == 0 and links:
                api_speed = sum(max(0, int(l.get("speed") or 0)) for l in links)

            speed_bps = api_speed if api_speed > 0 else self._computed_speed_bps
            speed_str = f"{format_size(speed_bps)}/s"

            # 6. Check if controller is stopped while unfinished links exist -> kick it
            unfinished_links = [l for l in links if not l.get("finished")]
            running_links = [l for l in links if l.get("running")]
            if unfinished_links and not running_links and wait_cycles % 2 == 0:
                self._kick_jd2_downloads(link_ids, package_ids)

            # Track finished links from API
            for l in links:
                if l.get("finished") and l.get("name"):
                    completed_files.add(l.get("name"))

            # --- CASE 1: Links not yet in DownloadList and no active .part file ---
            if not links and disk_loaded_bytes == 0:
                if len(completed_files) > 0 and wait_cycles >= 3:
                    return True, wait_cycles

                if wait_cycles % 2 == 0:
                    self._kick_jd2_downloads(link_ids, package_ids)

                self.progress_bar.bar_style = 'info'
                self.progress_bar.description = f"Connecting ({wait_cycles * 2}s)"
                self.status_label.value = (
                    f"<span style='color:#29b6f6;'>⏳ Đang chuyển link từ LinkGrabber sang Download & kết nối Hoster "
                    f"({wait_cycles * 2}s)... (Colab trống: <b>{cur_disk['free_gb']} GB</b>)</span>"
                )
                return False, wait_cycles

            # --- CASE 2: Active Download in Progress ---
            all_api_finished = bool(links) and all(bool(l.get("finished")) for l in links)
            has_part_file = any(
                f.endswith(".part") for f in (os.listdir(DRIVE_DOWNLOADS) if os.path.exists(DRIVE_DOWNLOADS) else [])
            )

            pct = (loaded_bytes / total_bytes * 100.0) if total_bytes > 0 else 0.0
            if all_api_finished and not has_part_file and len(completed_files) > 0:
                pct = 100.0

            eta_sec = int((total_bytes - loaded_bytes) / speed_bps) if (speed_bps > 0 and total_bytes > loaded_bytes) else 0
            eta_str = format_eta(eta_sec)

            curr_file = (
                (running_links[0].get("name") if running_links else None)
                or (unfinished_links[0].get("name") if unfinished_links else None)
                or disk_active_name
                or (links[0].get("name") if links else "File")
            )
            curr_status = (
                (running_links[0].get("status") if running_links else None)
                or (unfinished_links[0].get("status") if unfinished_links else None)
                or ("Downloading" if speed_bps > 0 or has_part_file else "Connecting...")
            )

            self.progress_bar.bar_style = 'info' if pct < 100.0 else 'success'
            self.progress_bar.value = max(0.0, min(100.0, pct))
            self.progress_bar.description = f"⬇️ {pct:.1f}% ({speed_str})"

            size_progress_str = (
                f"{format_size(loaded_bytes)} / {format_size(total_bytes)}"
                if total_bytes > 0 else f"{format_size(loaded_bytes)}"
            )

            self.status_label.value = (
                f"<div style='margin-top:4px; line-height:1.5;'>"
                f"⬇️ <b>Đang tải vào Google Drive:</b> <code>{curr_file}</code> [{curr_status}]<br>"
                f"📊 <b>Tiến độ:</b> <b>{pct:.1f}%</b> ({size_progress_str}) — "
                f"🚀 <b>Tốc độ:</b> <code>{speed_str}</code> — ⏱️ <b>ETA:</b> <code>{eta_str}</code> "
                f"<span style='color:#81c784;'>(Đĩa Colab trống: <b>{cur_disk['free_gb']} GB</b>)</span>"
                f"</div>"
            )

            if (all_api_finished or (not links and len(completed_files) > 0)) and not has_part_file and wait_cycles >= 2:
                return True, wait_cycles

        except Exception as loop_ex:
            self.status_label.value = f"<span style='color:#ffa000;'>⏳ Đang đồng bộ tiến trình tải: {str(loop_ex)[:80]}...</span>"

        return False, wait_cycles

    def _monitor_and_transfer(self, link_ids=None, package_ids=None, expected_names=None, raw_urls=None):
        """
        Background monitor thread that updates the ipywidgets progress bar in real time
        and guarantees restoring Resolve Links / Quick Download buttons upon exit or cancel.
        """
        self.is_downloading = True
        completed_files = set()
        wait_cycles = 0

        try:
            time.sleep(1.2)
            if not self._cancel_requested:
                self._kick_jd2_downloads(link_ids, package_ids)

            while self.is_downloading and not self._cancel_requested:
                time.sleep(1.5)
                if self._cancel_requested or not self.is_downloading:
                    break
                wait_cycles += 1
                done, wait_cycles = self._monitor_tick(
                    link_ids, package_ids, expected_names, completed_files, wait_cycles
                )
                if done:
                    break

        except Exception as err:
            if not self._cancel_requested:
                self.status_label.value = f"<span style='color:#e53935;'>⚠️ Lỗi theo dõi tiến trình: {str(err)[:100]}</span>"
        finally:
            self.is_downloading = False
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.btn_start.disabled = False
            self.disk_info_widget.value = self._get_disk_html()

            if not self._cancel_requested:
                self._scan_and_transfer_completed(completed_files)
                self.progress_bar.value = 100.0
                self.progress_bar.bar_style = 'success'
                self.progress_bar.description = "Done (100%)"
                final_disk = get_colab_disk_info()
                file_list_html = ", ".join(f"<code>{f}</code>" for f in sorted(completed_files)) if completed_files else "<code>MyDrive/Downloads/</code>"
                self.status_label.value = (
                    f"<div style='margin-top:8px; padding:10px; background-color:#1e3b2b; border:1px solid #2e7d32; border-radius:4px;'>"
                    f"<b>🎉 Hoàn thành tải file trực tiếp vào Google Drive!</b><br>"
                    f"📦 File đã lưu: {file_list_html}<br>"
                    f"📂 Thư mục: <code>/content/drive/MyDrive/Downloads/</code><br>"
                    f"<small style='color:#a5d6a7;'>⚡ Đĩa Colab cục bộ khả dụng: <b>{final_disk['free_gb']} GB</b> (Giữ nguyên không giảm).</small>"
                    f"</div>"
                )

    def _stream_direct_worker(self, urls: List[str]):
        """High-speed direct streaming worker for Yandex Disk links."""
        self.is_downloading = True
        completed = []
        try:
            for idx, url in enumerate(urls, 1):
                if self._cancel_requested or not self.is_downloading:
                    break
                self.status_label.value = f"<span style='color:#29b6f6;'>🔍 Đang giải mã liên kết tải nhanh [{idx}/{len(urls)}]: <code>{url[:60]}...</code></span>"
                yd_info = resolve_yandex_disk_direct_link(url) if "yandex" in url.lower() else None

                if not yd_info:
                    if self._ensure_service_ready():
                        self.jd.device.linkgrabber.add_links([{"autostart": True, "links": url, "destinationFolder": DRIVE_DOWNLOADS}])
                        self.jd.device.downloadcontroller.start_downloads()
                    continue

                direct_url, filename, filesize = yd_info
                safe_name = sanitize_filename(filename)
                dest_path = os.path.join(DRIVE_DOWNLOADS, safe_name)
                total_size_str = format_size(filesize)

                self.status_label.value = f"⬇️ Đang stream trực tiếp vào Google Drive: <b>{safe_name}</b> ({total_size_str})"
                start_t = time.time()
                bytes_done = 0
                last_t = start_t
                last_bytes = 0

                with requests.get(direct_url, stream=True, timeout=30) as resp:
                    resp.raise_for_status()
                    total_len = int(resp.headers.get('content-length', filesize))

                    with open(dest_path, "wb") as f_out:
                        for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
                            if not chunk or not self.is_downloading or self._cancel_requested:
                                break
                            f_out.write(chunk)
                            bytes_done += len(chunk)
                            curr_t = time.time()

                            if curr_t - last_t >= 1.0:
                                speed_bps = int((bytes_done - last_bytes) / (curr_t - last_t))
                                speed_str = f"{format_size(speed_bps)}/s"
                                pct = (bytes_done / total_len * 100.0) if total_len > 0 else 0.0
                                self.progress_bar.value = max(0.0, min(100.0, pct))
                                self.progress_bar.description = f"⬇️ {pct:.1f}% ({speed_str})"
                                self.disk_info_widget.value = self._get_disk_html()
                                cur_disk = get_colab_disk_info()
                                self.status_label.value = (
                                    f"⬇️ Streaming trực tiếp vào Google Drive: <b>{safe_name}</b> ({format_size(bytes_done)} / {total_size_str}) "
                                    f"— <code>{speed_str}</code> (Colab trống: <b>{cur_disk['free_gb']} GB</b>)"
                                )
                                last_t = curr_t
                                last_bytes = bytes_done

                if not self._cancel_requested:
                    completed.append(safe_name)

        except Exception as e:
            if not self._cancel_requested:
                self.status_label.value = f"<span style='color:#e53935;'>⚠️ Lỗi Streaming: {str(e)[:100]}</span>"
        finally:
            self.is_downloading = False
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.btn_start.disabled = False
            self.disk_info_widget.value = self._get_disk_html()
            if not self._cancel_requested:
                self.progress_bar.value = 100.0
                self.progress_bar.bar_style = 'success'
                self.progress_bar.description = "Done (100%)"
                final_disk = get_colab_disk_info()
                self.status_label.value = (
                    f"<div style='margin-top:8px; padding:10px; background-color:#2e3b2e; border-radius:4px;'>"
                    f"<b>🎉 Hoàn thành tải {len(completed)} file trực tiếp vào Google Drive!</b><br>"
                    f"📂 Thư mục lưu: <code>MyDrive/Downloads/</code><br>"
                    f"<small style='color:#a5d6a7;'>⚡ Đĩa Colab cục bộ khả dụng: <b>{final_disk['free_gb']} GB</b>.</small>"
                    f"</div>"
                )

    def _on_queue_up(self, b=None):
        selected = list(self.queue_list.value)
        if not selected:
            return
        indices = sorted([int(s.split('.')[0]) - 1 for s in selected])
        for idx in indices:
            if idx > 0 and idx - 1 not in indices:
                self.pending_items[idx], self.pending_items[idx - 1] = self.pending_items[idx - 1], self.pending_items[idx]
        self.update_queue_display()
        new_sel = [self.queue_list.options[max(0, i - 1)] for i in indices]
        self.queue_list.value = tuple(new_sel)

    def _on_queue_down(self, b=None):
        selected = list(self.queue_list.value)
        if not selected:
            return
        indices = sorted([int(s.split('.')[0]) - 1 for s in selected], reverse=True)
        for idx in indices:
            if idx < len(self.pending_items) - 1 and idx + 1 not in indices:
                self.pending_items[idx], self.pending_items[idx + 1] = self.pending_items[idx + 1], self.pending_items[idx]
        self.update_queue_display()
        new_sel = [self.queue_list.options[min(len(self.pending_items) - 1, i + 1)] for i in indices]
        self.queue_list.value = tuple(new_sel)

    def _on_queue_sort(self, b=None):
        self.pending_items.sort(key=lambda t: (t.get("name") or "").lower(), reverse=not self.sort_ascending)
        self.sort_ascending = not self.sort_ascending
        self.btn_sort.description = "Sort Z-A" if not self.sort_ascending else "Sort A-Z"
        self.update_queue_display()

    def _on_queue_select_all(self, b=None):
        self.queue_list.value = tuple(self.queue_list.options)

    def _on_queue_select_none(self, b=None):
        self.queue_list.value = ()

    def _on_queue_cancel(self, b=None):
        self.queue_container.layout.display = 'none'
        self.btn_resolve.disabled = False
        self.btn_quick.disabled = False
        self.status_label.value = "<span style='color:#aaa;'>Đã đóng danh sách chờ (Queue Preview).</span>"

    def _on_queue_remove(self, b=None):
        selected = list(self.queue_list.value)
        if not selected:
            return
        indices_to_remove = {int(s.split('.')[0]) - 1 for s in selected}
        removed_items = [self.pending_items[i] for i in indices_to_remove if 0 <= i < len(self.pending_items)]
        self.pending_items = [t for i, t in enumerate(self.pending_items) if i not in indices_to_remove]

        try:
            link_ids = [item.get("uuid") for item in removed_items if item.get("uuid")]
            package_ids = list(set([item.get("packageUUID") for item in removed_items if item.get("packageUUID")]))
            if link_ids or package_ids:
                self.jd.device.linkgrabber.remove_links(link_ids, package_ids)
        except Exception:
            pass

        self.update_queue_display()
        if not self.pending_items:
            self.queue_container.layout.display = 'none'
            self.status_label.value = "<span style='color:#aaa;'>Đã xóa toàn bộ mục khỏi danh sách chờ.</span>"

    def render(self):
        """Render the complete widget tree (without Sample Link button, with Cancel Download button)."""
        button_row = widgets.HBox([
            self.btn_resolve,
            self.btn_quick,
            self.btn_cancel_dl
        ], layout=widgets.Layout(margin='6px 0'))

        app_view = widgets.VBox([
            self.css_styles,
            self.disk_info_widget,
            self.cred_accordion,
            self.text_area,
            button_row,
            self.queue_container,
            self.progress_bar,
            self.status_label
        ], layout=widgets.Layout(width='100%', padding='10px'))

        try:
            display(app_view)
        except Exception:
            safe_print("ℹ️ Note: ipywidgets display failed in current environment.")


# --- DIRECT DRIVE STREAMING ENGINE ---
def resolve_yandex_disk_direct_link(yandex_url: str) -> Optional[Tuple[str, str, int]]:
    """Resolve public Yandex Disk URL to direct HTTPS CDN streaming link and metadata."""
    try:
        api_meta = f"https://cloud-api.yandex.net/v1/disk/public/resources?public_key={yandex_url.strip()}"
        res_meta = requests.get(api_meta, timeout=15)
        if res_meta.status_code != 200:
            return None
        data_meta = res_meta.json()
        filename = data_meta.get("name", "download.bin")
        filesize = data_meta.get("size", 0)

        api_dl = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={yandex_url.strip()}"
        res_dl = requests.get(api_dl, timeout=15)
        if res_dl.status_code != 200:
            return None
        download_url = res_dl.json().get("href")
        if download_url:
            return (download_url, filename, filesize)
    except Exception:
        pass
    return None


# --- MAIN RUNNER ---
def run_app():
    """Khởi chạy ứng dụng Colab Downloader (hiển thị 1 ô giao diện duy nhất)."""
    clear_output(wait=True)
    safe_print("🚀 CloudToDrive v2.2 — JDownloader 2 Headless (Auto-Confirm & Auto-Start) đã sẵn sàng!")
    app = ColabDownloaderApp()
    app.render()
    return None


if __name__ == "__main__":
    run_app()